# CrewAI: 멀티 에이전트 협업 프레임워크

이번 노트북에서는 **CrewAI**의 핵심 개념을 학습하고, 프로젝트를 만드는 5단계 워크플로우를 따라가며 직접 에이전트를 구성해봅니다.

## 개요

| 주제 | 내용 |
|------|------|
| CrewAI 소개 | 역할 기반 멀티 에이전트 프레임워크 핵심 개념 |
| 핵심 구성요소 | Agent, Task, Crew, Process |
| 프로젝트 5단계 | create → config → crew.py → main.py → run |
| 설정 방식 | YAML 기반 vs Python 코드 기반 에이전트/태스크 정의 |
| 데코레이터 패턴 | @CrewBase, @agent, @task, @crew |
| Custom Tool | BaseTool 상속을 통한 커스텀 도구 작성 |

## 학습 목표

1. CrewAI의 핵심 개념(Agent, Task, Crew, Process)을 이해하기
2. CrewAI CLI로 프로젝트를 생성하고 5단계로 완성하는 워크플로우 익히기
3. YAML과 Python 코드로 에이전트와 태스크를 정의하는 두 가지 방식 비교하기
4. 데코레이터 패턴(`@CrewBase`, `@agent`, `@task`, `@crew`)으로 프로젝트 구조화하기
5. BaseTool을 상속하여 커스텀 도구를 만드는 방법 이해하기

---

## 다른 프레임워크와의 비교

| 특성 | CrewAI | OpenAI Agents SDK | LangGraph |
|------|--------|-------------------|----------|
| **핵심 철학** | 역할 기반 협업 | 도구 + 핸드오프 | 그래프 기반 워크플로우 |
| **에이전트 정의** | role, goal, backstory | instructions | 노드(함수) |
| **실행 흐름** | Process (sequential/hierarchical) | Runner.run() | 상태 그래프 |
| **설정 방식** | YAML + 데코레이터 | Python 코드 | Python 코드 |
| **협업 패턴** | Crew가 자동 조율 | Handoff / as_tool | Edge로 명시적 연결 |
| **추상화 수준** | 높음 (선언적) | 중간 | 낮음 (명시적) |

---

## 1. CrewAI 핵심 개념

CrewAI는 **역할(Role) 기반** 멀티 에이전트 프레임워크입니다. 실제 팀처럼 각 에이전트에게 역할, 목표, 배경 스토리를 부여하고, 태스크를 할당하여 협업하게 합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CrewAI 핵심 구성 요소                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────┐   role + goal + backstory를 가진 AI 에이전트          │
│  │  Agent   │   각 에이전트는 고유한 역할과 전문성을 보유           │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트가 수행할 구체적인 작업 단위                 │
│  │  Task    │   description + expected_output + agent 할당          │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트들과 태스크들을 묶는 팀                      │
│  │  Crew    │   실행 프로세스와 전체 워크플로우를 관리              │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   태스크 실행 순서를 결정하는 방식                     │
│  │ Process  │   sequential(순차) / hierarchical(계층적)            │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트가 사용할 수 있는 외부 도구                  │
│  │  Tool    │   BaseTool 상속 또는 @tool 데코레이터                │
│  └─────────┘                                                       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Process 타입

```
Sequential (순차 실행):     Task A ──▶ Task B ──▶ Task C
                            이전 태스크의 출력이 다음 태스크의 컨텍스트로 전달

Hierarchical (계층적 실행): Manager Agent가 태스크를 동적으로 분배
                                  ┌──▶ Task A (Agent 1)
                            Manager├──▶ Task B (Agent 2)
                                  └──▶ Task C (Agent 3)
```

### Agent vs Task — 무엇이 다른가?

처음에 헷갈리기 쉬운 부분입니다. 회사에 비유하면 명확해집니다.

```
Agent = "누가" (사람)          Task = "무엇을" (업무 지시서)
━━━━━━━━━━━━━━━━━━━━━        ━━━━━━━━━━━━━━━━━━━━━━━━━━━
role: 시니어 개발자             description: 로그인 API를 구현하세요
goal: 안정적인 코드 작성        expected_output: REST API 엔드포인트
backstory: 10년차 백엔드 전문가  agent: 시니어 개발자 ← 누가 할 것인가
```

**핵심 차이점**:

| | Agent (사람) | Task (업무 지시서) |
|---|---|---|
| **정의** | "나는 누구인가" | "무엇을 해야 하는가" |
| **속성** | role, goal, backstory | description, expected_output |
| **재사용** | 하나의 Agent가 여러 Task 수행 가능 | 하나의 Task는 하나의 Agent에 할당 |
| **비유** | 직원 (채용) | 업무 티켓 (JIRA) |

**예시 — 스타트업 블로그 팀**:

```
┌─────────────── Agent (사람) ───────────────┐
│                                             │
│  김연구 (researcher)                        │
│    role: "AI 기술 연구원"                   │
│    goal: "최신 트렌드를 조사하고 정리"       │
│    backstory: "10년 경력 연구원"            │
│                                             │
│  박작가 (writer)                            │
│    role: "기술 블로그 작가"                  │
│    goal: "쉬운 블로그 글 작성"               │
│    backstory: "기술 콘텐츠 전문가"           │
│                                             │
└─────────────────────────────────────────────┘

┌─────────────── Task (업무 지시서) ──────────┐
│                                             │
│  조사 업무 (research_task)                   │
│    description: "AI 에이전트 트렌드를 조사"  │
│    expected_output: "핵심 포인트 3-5개"      │
│    agent: 김연구  ← 이 사람이 수행           │
│                                             │
│  글쓰기 업무 (write_task)                    │
│    description: "트렌드 블로그 글 작성"      │
│    expected_output: "500자 블로그 글"        │
│    agent: 박작가  ← 이 사람이 수행           │
│                                             │
│  편집 업무 (edit_task)                       │
│    description: "블로그 글 교정 및 다듬기"   │
│    expected_output: "최종 완성본"            │
│    agent: 박작가  ← 같은 사람이 다른 업무!   │
│                                             │
└─────────────────────────────────────────────┘
```

위 예시에서 **박작가(writer)**는 글쓰기와 편집, 두 개의 Task를 수행합니다. Agent는 "사람"이고 Task는 "할 일"이기 때문에, 한 사람이 여러 업무를 맡을 수 있는 것입니다.

---

## 2. CrewAI 프로젝트 만들기 - 5단계 워크플로우

CrewAI는 CLI를 제공하여 프로젝트 스캐폴딩부터 실행까지 체계적인 워크플로우를 지원합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│              CrewAI 프로젝트 5단계 워크플로우                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Step 1.  crewai create crew my_project                            │
│           │  프로젝트 스캐폴딩 생성                                 │
│           ▼                                                         │
│  Step 2.  config/agents.yaml, config/tasks.yaml 작성               │
│           │  에이전트 역할/목표/배경, 태스크 설명/기대출력 정의      │
│           ▼                                                         │
│  Step 3.  crew.py 완성                                              │
│           │  @agent, @task, @crew 데코레이터로 config 연결          │
│           ▼                                                         │
│  Step 4.  main.py 업데이트                                          │
│           │  inputs 설정 및 kickoff() 호출                          │
│           ▼                                                         │
│  Step 5.  crewai run                                                │
│           실행!                                                     │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Step 1. 프로젝트 생성

```bash
crewai create crew my_project
```

이 명령어가 생성하는 프로젝트 구조:

```
my_project/
├── pyproject.toml              # 프로젝트 설정 (crewai 의존성)
├── README.md
├── knowledge/                  # Knowledge 소스 (RAG용, 선택사항)
│   └── user_preference.txt
├── output/                     # 태스크 결과물 저장 디렉토리
└── src/my_project/
    ├── __init__.py
    ├── config/
    │   ├── agents.yaml         # ← Step 2에서 작성
    │   └── tasks.yaml          # ← Step 2에서 작성
    ├── tools/
    │   └── custom_tool.py      # 커스텀 도구 템플릿
    ├── crew.py                 # ← Step 3에서 완성
    └── main.py                 # ← Step 4에서 업데이트
```

### Step 2. YAML 설정 파일 작성

에이전트와 태스크를 선언적으로 정의합니다. `{variable}` 형태의 변수는 실행 시 `inputs`로 전달됩니다.

**config/agents.yaml** — 에이전트의 역할, 목표, 배경 스토리 정의:

```yaml
researcher:
  role: >
    Senior Research Analyst on {topic}
  goal: >
    Uncover cutting-edge developments in {topic}
  backstory: >
    You're a seasoned researcher with a knack for
    uncovering the latest developments in {topic}.
  llm: openai/gpt-4o-mini

writer:
  role: >
    Tech Content Strategist
  goal: >
    Craft compelling content on {topic}
  backstory: >
    You're a renowned content strategist known for
    making complex tech topics accessible and engaging.
  llm: openai/gpt-4o-mini
```

**config/tasks.yaml** — 태스크의 설명, 기대 출력, 담당 에이전트 정의:

```yaml
research_task:
  description: >
    Conduct thorough research about {topic}.
    Identify key trends and breakthrough technologies.
  expected_output: >
    A comprehensive report with the latest developments.
  agent: researcher

writing_task:
  description: >
    Using the research findings, write an engaging blog post about {topic}.
  expected_output: >
    A 4-paragraph blog post formatted in markdown.
  agent: writer
  output_file: output/blog_post.md
```

### Step 3. crew.py 완성

데코레이터로 YAML 설정과 코드를 연결합니다:

```python
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task

@CrewBase                           # YAML 설정 자동 로드
class MyProject():
    agents_config = 'config/agents.yaml'
    tasks_config = 'config/tasks.yaml'

    @agent                          # self.agents 리스트에 자동 추가
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'],
            verbose=True
        )

    @agent
    def writer(self) -> Agent:
        return Agent(
            config=self.agents_config['writer'],
            verbose=True
        )

    @task                           # self.tasks 리스트에 자동 추가
    def research_task(self) -> Task:
        return Task(config=self.tasks_config['research_task'])

    @task
    def writing_task(self) -> Task:
        return Task(config=self.tasks_config['writing_task'])

    @crew                           # 최종 Crew 조립
    def crew(self) -> Crew:
        return Crew(
            agents=self.agents,     # @agent 메서드들이 자동 수집됨
            tasks=self.tasks,       # @task 메서드들이 자동 수집됨
            process=Process.sequential,
            verbose=True,
        )
```

### Step 4. main.py 업데이트

```python
from my_project.crew import MyProject

def run():
    inputs = {
        'topic': 'AI Agents in 2025'
    }
    result = MyProject().crew().kickoff(inputs=inputs)
    print(result.raw)
```

`inputs`의 `topic` 값이 YAML의 모든 `{topic}` 자리에 자동으로 삽입됩니다.

### Step 5. 실행

```bash
crewai run
```

### 데코레이터 역할 정리

| 데코레이터 | 역할 |
|-----------|------|
| `@CrewBase` | YAML 파일 자동 로드, `self.agents`/`self.tasks` 자동 생성 |
| `@agent` | 메서드를 에이전트로 등록 → `self.agents` 리스트에 자동 추가 |
| `@task` | 메서드를 태스크로 등록 → `self.tasks` 리스트에 자동 추가 |
| `@crew` | 최종 Crew 객체를 조립하는 메서드 지정 |

---

## 3. 환경 설정

In [ ]:
# CrewAI 설치 (처음 한 번만 실행)
# 터미널에서: pip install 'crewai[tools]'
# 또는 노트북에서:
# import sys
# !{sys.executable} -m pip install 'crewai[tools]'

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="pysbd")

from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

In [ ]:
from crewai import Agent, Task, Crew, Process

print("CrewAI import 성공!")

---

## 4. Agent 정의 — Python 코드 방식

YAML 없이 Python 코드에서 직접 Agent를 생성할 수 있습니다. 노트북에서 빠르게 실험하기에 적합합니다.

Agent의 3대 속성:
- **role**: 에이전트의 역할 (예: 연구원, 작가, 편집자)
- **goal**: 에이전트가 달성해야 할 목표
- **backstory**: 에이전트의 배경 스토리 (행동 성향에 영향)

In [ ]:
# Python 코드로 직접 Agent 생성하기

researcher = Agent(
    role="AI 기술 연구원",
    goal="최신 AI 기술 트렌드를 조사하고 핵심 내용을 정리한다",
    backstory="당신은 10년 경력의 AI 연구원으로, 복잡한 기술 개념을 쉽게 설명하는 능력이 뛰어납니다.",
    verbose=True,
    llm="openai/gpt-4o-mini",
)

writer = Agent(
    role="기술 블로그 작가",
    goal="연구 결과를 바탕으로 이해하기 쉬운 블로그 글을 작성한다",
    backstory="당신은 기술 블로그 전문 작가로, 전문적인 내용을 일반 독자도 이해할 수 있게 풀어쓰는 재능이 있습니다.",
    verbose=True,
    llm="openai/gpt-4o-mini",
)

print(f"Agent 생성 완료: {researcher.role}, {writer.role}")

---

## 5. Task 정의 — Python 코드 방식

Task는 에이전트가 수행할 구체적인 작업입니다.

Task의 핵심 속성:
- **description**: 수행할 작업에 대한 상세 설명
- **expected_output**: 기대하는 출력 형태
- **agent**: 이 태스크를 수행할 에이전트
- **output_file** (선택): 결과를 파일로 저장할 경로

In [ ]:
# Python 코드로 직접 Task 생성하기

research_task = Task(
    description="2024-2025년 AI 에이전트 프레임워크의 주요 트렌드를 조사하고 핵심 내용을 정리하세요.",
    expected_output="AI 에이전트 프레임워크 트렌드 요약 (3-5개 핵심 포인트)",
    agent=researcher,
)

write_task = Task(
    description="연구 결과를 바탕으로 'AI 에이전트 프레임워크 트렌드'에 대한 짧은 블로그 글을 작성하세요.",
    expected_output="500자 내외의 블로그 글",
    agent=writer,
)

print(f"Task 생성 완료: {len([research_task, write_task])}개")

---

## 6. Crew 구성 및 실행

Agent와 Task를 묶어 Crew를 구성하고 실행합니다. `Process.sequential`은 태스크를 순서대로 실행하며, 이전 태스크의 출력이 다음 태스크의 컨텍스트로 자동 전달됩니다.

```
┌──────────────────────────────────────────────────────────┐
│                    Crew 실행 흐름                         │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  crew.kickoff()                                          │
│       │                                                  │
│       ▼                                                  │
│  ┌─────────────┐    결과 전달    ┌─────────────┐        │
│  │ research_task│ ─────────────▶ │  write_task  │        │
│  │ (researcher) │               │   (writer)   │        │
│  └─────────────┘               └─────────────┘         │
│                                       │                  │
│                                       ▼                  │
│                                  CrewOutput              │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
# Crew 구성 및 실행

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()

print("\n" + "="*60)
print("최종 결과:")
print("="*60)
print(result.raw)

---

## 7. Custom Tool 만들기

CrewAI에서는 `BaseTool`을 상속하여 에이전트가 사용할 커스텀 도구를 만들 수 있습니다.

### 커스텀 도구 작성 패턴

```python
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

class MyToolInput(BaseModel):           # 1. 입력 스키마 정의
    argument: str = Field(..., description="...")

class MyTool(BaseTool):                 # 2. BaseTool 상속
    name: str = "도구 이름"              # 3. 에이전트가 식별할 이름
    description: str = "도구 설명"       # 4. 에이전트가 사용 시점을 판단
    args_schema: Type[BaseModel] = MyToolInput

    def _run(self, argument: str) -> str:  # 5. 실제 로직 구현
        return "result"
```

| 구성 요소 | 설명 |
|----------|------|
| `Input Schema` | Pydantic BaseModel로 입력 스키마 정의 (에이전트가 파라미터를 이해) |
| `name` | 도구 이름 (에이전트가 도구를 식별) |
| `description` | 도구 설명 (에이전트가 언제 이 도구를 사용할지 판단) |
| `args_schema` | Input Schema 클래스 연결 |
| `_run()` | 실제 도구 로직 구현 |

In [ ]:
# 커스텀 도구 예제: 간단한 단어 수 세기 도구

from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field


class WordCountInput(BaseModel):
    """Input schema for WordCountTool."""
    text: str = Field(..., description="The text to count words in.")


class WordCountTool(BaseTool):
    name: str = "Word Counter"
    description: str = "Counts the number of words in a given text. Useful when you need to check text length."
    args_schema: Type[BaseModel] = WordCountInput

    def _run(self, text: str) -> str:
        word_count = len(text.split())
        return f"The text contains {word_count} words."


# 도구를 사용하는 Agent 생성
word_counter = WordCountTool()

editor = Agent(
    role="편집자",
    goal="글의 단어 수를 확인하고 피드백을 제공한다",
    backstory="당신은 꼼꼼한 편집자로, 항상 글의 길이를 체크합니다.",
    tools=[word_counter],
    verbose=True,
    llm="openai/gpt-4o-mini",
)

print(f"도구가 할당된 Agent 생성: {editor.role}")
print(f"사용 가능한 도구: {[t.name for t in editor.tools]}")

---

## 8. Context — 태스크 간 정보 전달

CrewAI에서 태스크들이 서로 협업하려면 이전 태스크의 결과를 참고할 수 있어야 합니다. 이 역할을 하는 것이 **Context**입니다.

### Context가 전달되는 두 가지 방식

**방식 1: Sequential 프로세스의 자동 전달**

`Process.sequential`에서는 태스크 리스트 순서대로 실행되며, **이전 태스크의 출력이 자동으로 다음 태스크의 컨텍스트**가 됩니다. 별도 설정이 필요 없습니다.

```
tasks = [research_task, write_task, edit_task]

research_task 실행
     │
     │ 출력이 자동으로 context에 포함
     ▼
write_task 실행  ← research_task의 결과를 참고하여 작성
     │
     │ 출력이 자동으로 context에 포함
     ▼
edit_task 실행   ← write_task의 결과를 참고하여 편집
```

**방식 2: `context` 파라미터로 명시적 지정**

특정 태스크의 결과만 선택적으로 전달하고 싶을 때 `context` 파라미터를 사용합니다. 순서에 상관없이 원하는 태스크의 출력을 직접 연결할 수 있습니다.

```python
# edit_task가 research_task와 write_task 두 결과를 모두 참고
edit_task = Task(
    description="블로그 글을 교정하세요.",
    expected_output="최종 완성본",
    agent=editor,
    context=[research_task, write_task],   # ← 명시적으로 지정
)
```

### 비유로 이해하기

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Context = 업무 참고 자료                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Sequential 자동 전달:                                              │
│    마치 릴레이 경주처럼, 앞 주자의 바통(결과)이                     │
│    자동으로 다음 주자에게 넘어갑니다.                                │
│                                                                     │
│    Task A ──바통──▶ Task B ──바통──▶ Task C                         │
│                                                                     │
│  명시적 context 지정:                                               │
│    마치 회의에서 "이 보고서와 저 자료를 참고해서 작성해주세요"       │
│    라고 특정 자료를 지목하는 것과 같습니다.                          │
│                                                                     │
│    Task A ──────────────────┐                                       │
│                              ├──▶ Task C (context=[A, B])           │
│    Task B ──────────────────┘                                       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Sequential 자동 전달 vs 명시적 context

| | Sequential 자동 전달 | 명시적 `context` 지정 |
|---|---|---|
| **설정** | 없음 (자동) | `context=[task_a, task_b]` |
| **전달 범위** | 바로 직전 태스크의 출력만 | 원하는 태스크를 자유롭게 선택 |
| **사용 시점** | 단순한 파이프라인 | 여러 태스크 결과를 조합할 때 |
| **비유** | 릴레이 바통 | 참고 자료 지정 |

> **Debate 프로젝트에서의 Context**: sequential 프로세스이므로 oppose 태스크는 propose의 결과를 자동으로 받고, decide 태스크는 oppose의 결과(+ 이전 컨텍스트)를 받아 양쪽 논거를 모두 참고하여 판결합니다.

---

## 정리

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CrewAI 핵심 요약                                 │
├──────────────────────────────┬──────────────────────────────────────┤
│     개념                     │     핵심 포인트                      │
├──────────────────────────────┼──────────────────────────────────────┤
│  Agent                       │  role + goal + backstory로 정의      │
│  Task                        │  description + expected_output       │
│  Crew                        │  agents + tasks + process            │
│  Process                     │  sequential / hierarchical           │
│  Tool                        │  BaseTool 상속, _run() 구현          │
│  데코레이터                  │  @CrewBase, @agent, @task, @crew     │
│  설정 방식                   │  YAML 선언적 / Python 코드 직접      │
└──────────────────────────────┴──────────────────────────────────────┘
```

### 프로젝트 5단계 요약

```bash
# Step 1. 프로젝트 생성
crewai create crew my_project

# Step 2. YAML 설정 파일 작성
#   → config/agents.yaml, config/tasks.yaml

# Step 3. crew.py 완성
#   → @agent, @task, @crew 데코레이터로 config 연결

# Step 4. main.py 업데이트
#   → inputs 설정 및 kickoff() 호출

# Step 5. 실행
crewai run
```

### 다음 노트북

다음 노트북(03-2)에서는 실제 CrewAI 프로젝트인 **Debate(찬반 토론)** 샘플을 분석하고 직접 실행합니다.